[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S1_recta_vs_polinomial.ipynb)

# Demo: recta contra polinomial

Demo en vivo de S1, unos 15 minutos. No se entrega. Somos el equipo de análisis y tenemos dos preguntas:
cuánto rinde un auto según su potencia y cuántas bicis salen de la estación 51 según la hora.
Primero probamos una recta, la medimos, miramos dónde falla y luego le damos curva.

Todo con scikit-learn, pandas y seaborn. Corre las celdas en orden; las dos primeras preparan el entorno
hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from dammg_utils import COLORES

du.estilo_dammg()

## 1. Los dos datos

**Autos:** 392 autos de los años 70 y 80, con su potencia (`horsepower`) y su rendimiento en millas por galón (`mpg`).
Separamos 75 % para ajustar (train) y 25 % para validar, con `train_test_split`.

**MiBici:** la estación 51 de lunes a viernes, salidas por hora (`departures`) contra la hora del día (`hour_of_day`).
Ajustamos con train (octubre a junio) y validamos con julio.

**¿Cuántos datos tenemos para ajustar y cuántos para validar?**

In [ ]:
def externo(nombre):
    """Copia local del repo si existe; si no, sns.load_dataset la baja de seaborn-data."""
    import pathlib
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        if (p / "data" / "externos" / f"{nombre}.csv").is_file():
            return pd.read_csv(p / "data" / "externos" / f"{nombre}.csv")
    return sns.load_dataset(nombre)


autos = externo("mpg").dropna(subset=["horsepower", "mpg"])
autos_train, autos_val = train_test_split(autos, test_size=0.25, random_state=0)

df = pd.read_parquet(RUTA_DATOS)
e51 = df[(df.station_id == 51) & (df.is_weekend == 0)]
bici_train = e51[e51.particion == "train"]
bici_val = e51[e51.particion == "val"]

pd.DataFrame({"train": [len(autos_train), len(bici_train)], "validación": [len(autos_val), len(bici_val)]},
             index=["autos", "MiBici 51"])

**¿Qué esperamos ver?** 294 autos para ajustar y 98 para validar. En MiBici, 4,537 horas de train y 552 de julio.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
sns.scatterplot(data=autos_train, x="horsepower", y="mpg", color=COLORES["w"], alpha=0.6, ax=axs[0])
axs[0].set(title="Autos (train)", xlabel="caballos de fuerza", ylabel="millas por galón")
sns.scatterplot(data=bici_train, x="hour_of_day", y="departures", color=COLORES["w"], alpha=0.1, ax=axs[1])
sns.lineplot(data=bici_train, x="hour_of_day", y="departures", color=COLORES["tinta"], marker="o", ax=axs[1])
axs[1].set(title="Estación 51, entre semana (train)", xlabel="hora del día", ylabel="salidas por hora")
fig.tight_layout()
plt.show()

**¿Qué esperamos ver?** En autos, a más potencia menos rendimiento, pero bajando como curva: cae rápido al principio
y luego se aplana. En MiBici, casi nada de madrugada, un pico fuerte en la mañana y otro en la tarde (la línea negra es el promedio de cada hora).
Ninguno de los dos se ve como recta.

## 2. La recta

Primero los autos. `LinearRegression` busca la recta $\hat y = w\,x + b$ con el menor error cuadrático en train.

**¿Cuánto rendimiento se pierde por cada caballo de fuerza extra?**

In [ ]:
recta = LinearRegression().fit(autos_train[["horsepower"]], autos_train.mpg)
w, b = recta.coef_[0], recta.intercept_
print(f"pendiente w = {w:.3f}   intercepto b = {b:.1f}")
print(f"En palabras: cada caballo de fuerza extra baja el rendimiento {-w:.2f} millas por galón; "
      f"100 caballos más, {-100 * w:.1f} millas menos.")
print(f"Un auto de 0 caballos rendiría {b:.1f} millas por galón: no existe, el intercepto solo ancla la recta.")

**¿Qué esperamos ver?** Pendiente de -0.16: cada caballo de fuerza extra quita 0.16 millas por galón, o 16 millas por cada 100 caballos.
El intercepto, 40 millas por galón, es el rendimiento de un auto sin motor: sirve para anclar la recta, no para interpretarlo.

**¿Cómo se ve esa recta sobre los puntos?**

In [ ]:
malla = pd.DataFrame({"horsepower": range(int(autos.horsepower.min()), int(autos.horsepower.max()) + 1)})
malla["predicción"] = recta.predict(malla[["horsepower"]])

ax = sns.scatterplot(data=autos_train, x="horsepower", y="mpg", color=COLORES["w"], alpha=0.6)
sns.lineplot(data=malla, x="horsepower", y="predicción", color=COLORES["error"], linewidth=3, ax=ax)
ax.set(title="Autos: la recta de LinearRegression", xlabel="caballos de fuerza", ylabel="millas por galón")
plt.show()

**¿Qué esperamos ver?** La recta pasa por el centro de la nube, pero en las orillas se despega: en potencias bajas
los autos rinden más de lo que dice la recta y en potencias altas la recta baja de más.

Ahora las tres varas: **MAE** (el error típico), **RMSE** (igual, pero castiga más los errores grandes) y
**R²** (cuánto mejora sobre decir siempre el promedio: 0 es igual, 1 es perfecto). Las medimos en train y en validación.

**¿El error en validación es mucho peor que en train?**

In [ ]:
def varas(modelo, X_train, y_train, X_val, y_val):
    """MAE, RMSE y R² del modelo en train y en validación, en una tabla."""
    filas = {}
    for nombre, X, y in [("train", X_train, y_train), ("validación", X_val, y_val)]:
        y_hat = modelo.predict(X)
        filas[nombre] = {"MAE": mean_absolute_error(y, y_hat),
                         "RMSE": root_mean_squared_error(y, y_hat),
                         "R²": r2_score(y, y_hat)}
    return pd.DataFrame(filas).T.round(2)


varas(recta, autos_train[["horsepower"]], autos_train.mpg, autos_val[["horsepower"]], autos_val.mpg)

**¿Qué esperamos ver?** MAE de 3.9 en train y 3.6 en validación: la recta se equivoca por unas 3.5 a 4 millas por galón.
R² de 0.60 y 0.62. Validación sale un poco mejor que train por suerte de la partición; lo importante es que no se desploma.

### ¿Qué es el R²?

El R² compara tu modelo contra el modelo más tonto posible: **decir siempre el promedio**.

$$R^2 = 1 - \frac{\text{error al cuadrado de tu modelo}}{\text{error al cuadrado de decir siempre el promedio}}$$

- $R^2 = 1$: tu modelo no se equivoca nunca.
- $R^2 = 0$: tu modelo es igual de malo que decir siempre el promedio.
- $R^2 < 0$: tu modelo es **peor** que decir siempre el promedio.

**Antes de correr: ¿el error de la recta será mucho más chico que el de decir siempre el promedio?**

In [ ]:
y_val = autos_val.mpg
promedio_train = autos_train.mpg.mean()

error_recta = ((recta.predict(autos_val[["horsepower"]]) - y_val) ** 2).sum()
error_promedio = ((promedio_train - y_val) ** 2).sum()

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
sns.scatterplot(data=autos_val, x="horsepower", y="mpg", color=COLORES["w"], alpha=0.6, ax=axs[0])
orden = autos_val.sort_values("horsepower")
axs[0].plot(orden.horsepower, recta.predict(orden[["horsepower"]]), color=COLORES["error"], label="la recta")
axs[0].axhline(promedio_train, color="gray", linestyle="--", label=f"siempre el promedio ({promedio_train:.1f})")
axs[0].legend()
axs[0].set_title("Dos modelos para los mismos autos")
sns.barplot(x=["decir siempre el promedio", "la recta"], y=[error_promedio, error_recta],
            hue=["decir siempre el promedio", "la recta"], palette=["gray", COLORES["error"]], legend=False, ax=axs[1])
axs[1].set_ylabel("suma de errores al cuadrado")
axs[1].set_title("¿Cuánto error le quitó la recta al promedio?")
plt.tight_layout()
plt.show()

print(f"error del promedio: {error_promedio:,.0f}   error de la recta: {error_recta:,.0f}")
print(f"R² = 1 − {error_recta:,.0f} / {error_promedio:,.0f} = {1 - error_recta / error_promedio:.2f}")
print(f"con r2_score: {r2_score(y_val, recta.predict(autos_val[['horsepower']])):.2f}")

**¿Qué esperamos ver?** La barra roja es mucho más chica que la gris: la recta le quita cerca del 60 % del error
a decir siempre el promedio, y eso es su R². El cálculo a mano y `r2_score` dan lo mismo.

**¿Y un R² negativo?** Si un modelo es peor que el promedio, su barra queda más alta que la gris, la fracción pasa de 1
y el R² sale negativo. Pasa, por ejemplo, con un polinomio de grado muy alto ajustado con pocos datos: en el widget del grado de las slides, la validación llega a R² negativo.

Lo mismo con MiBici: una recta de salidas contra hora.

**¿Una recta puede seguir dos picos?**

In [ ]:
recta_bici = LinearRegression().fit(bici_train[["hour_of_day"]], bici_train.departures)
print(f"pendiente w = {recta_bici.coef_[0]:.3f}   intercepto b = {recta_bici.intercept_:.2f}")
print(f"En palabras: cada hora que pasa suma {recta_bici.coef_[0]:.2f} salidas, de las 0 a las 23 h, sin importar el pico.")
varas(recta_bici, bici_train[["hour_of_day"]], bici_train.departures, bici_val[["hour_of_day"]], bici_val.departures)

**¿Qué esperamos ver?** Pendiente de 0.36 salidas por hora y R² de 0.07 en train y 0.00 en julio: la recta es casi igual que decir siempre
el promedio. Se equivoca por unas 7 bicis en train y 8.5 en julio, cuando en la hora pico salen 26 a 29.

## 3. Los residuos

El residuo es lo que la recta no explica: $y - \hat y$. Si la recta capturó todo, los residuos son ruido
alrededor de cero, sin forma. `sns.residplot` ajusta la misma recta y dibuja sus residuos.

**¿Los residuos se ven como ruido o tienen forma?**

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
sns.residplot(data=autos_train, x="horsepower", y="mpg", color=COLORES["w"], scatter_kws={"alpha": 0.6}, ax=axs[0])
axs[0].set(title="Autos: residuos de la recta", xlabel="caballos de fuerza", ylabel="mpg real menos predicho")
sns.residplot(data=bici_train, x="hour_of_day", y="departures", color=COLORES["w"], scatter_kws={"alpha": 0.1}, ax=axs[1])
axs[1].set(title="MiBici 51: residuos de la recta", xlabel="hora del día", ylabel="salidas reales menos predichas")
fig.tight_layout()
plt.show()

**¿Qué esperamos ver?** En autos, una sonrisa: residuos positivos en las orillas y negativos en medio.
En MiBici, los dos picos siguen ahí, ahora alrededor de cero. Los residuos tienen forma: a la recta le falta curva.

## 4. Polinomial: la misma regresión lineal con columnas nuevas

`PolynomialFeatures` no cambia el modelo, cambia las columnas: de $x$ hace $1, x, x^2, x^3, \dots$
y `LinearRegression` le pone un peso a cada una. Lo juntamos todo en un `make_pipeline`.

**¿Qué columnas crea para tres autos?**

In [ ]:
tres = autos_train[["horsepower"]].head(3)
columnas = PolynomialFeatures(degree=3).set_output(transform="pandas").fit_transform(tres)
display(columnas)

**¿Qué esperamos ver?** Cuatro columnas: el 1 (para el intercepto), la potencia, su cuadrado y su cubo. Un auto de 198 caballos
da 7,762,392 en la columna del cubo. Con grado 8 esos números se vuelven enormes y la cuenta pierde precisión,
por eso el pipeline primero escala $x$ a $[-1, 1]$ con `MinMaxScaler` y después crea las columnas.

In [ ]:
def polinomial(d):
    """Escala x a [-1, 1], crea las columnas hasta x^d y ajusta una regresión lineal."""
    return make_pipeline(MinMaxScaler(feature_range=(-1, 1)), PolynomialFeatures(degree=d), LinearRegression())


DATOS = {
    "Autos": (autos_train, autos_val, "horsepower", "mpg"),
    "MiBici 51": (bici_train, bici_val, "hour_of_day", "departures"),
}

**¿Cómo cambia la curva con grado 1, 2, 3 y 8?**

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(17, 7.5))
for fila, (nombre, (train, val, x, y)) in zip(axs, DATOS.items()):
    malla = pd.DataFrame({x: range(int(train[x].min()), int(train[x].max()) + 1)})
    for ax, d in zip(fila, [1, 2, 3, 8]):
        modelo = polinomial(d).fit(train[[x]], train[y])
        malla["predicción"] = modelo.predict(malla[[x]])
        rmse_val = root_mean_squared_error(val[y], modelo.predict(val[[x]]))
        sns.scatterplot(data=train, x=x, y=y, color=COLORES["w"], alpha=0.15, ax=ax, legend=False)
        sns.lineplot(data=malla, x=x, y="predicción", color=COLORES["error"], linewidth=3, ax=ax)
        ax.set(title=f"{nombre}, grado {d}: RMSE val {rmse_val:.2f}", xlabel=x, ylabel=y)
fig.tight_layout()
plt.show()

**¿Qué esperamos ver?** En autos, el grado 2 ya dobla la curva y el RMSE de validación baja de 4.74 a 4.03; los grados 3 y 8 casi no cambian
el número, solo se ponen nerviosos en las orillas, donde hay pocos autos. En MiBici, grado 1 y 2 no ven los picos;
grado 3 da lo mismo que 2 (10.70); con grado 8 ya aparecen la mañana y la tarde, y el RMSE de julio baja de 11.78 a 8.36.

## 5. Las tres varas por grado

Ajustamos los grados 1 a 8 y medimos MAE, RMSE y R² en train y en validación.

**¿Más grado siempre es mejor?**

In [ ]:
filas = []
for nombre, (train, val, x, y) in DATOS.items():
    for d in range(1, 9):
        modelo = polinomial(d).fit(train[[x]], train[y])
        tabla = varas(modelo, train[[x]], train[y], val[[x]], val[y])
        for particion, fila in tabla.iterrows():
            filas.append({"datos": nombre, "grado": d, "partición": particion, **fila})
por_grado = pd.DataFrame(filas)

display(por_grado.pivot_table(index="grado", columns=["datos", "partición"], values="RMSE").round(2))

**¿Qué esperamos ver?** En train el RMSE nunca sube: autos va de 4.94 a 4.34 y MiBici de 9.30 a 6.38.
En validación es otra historia. Autos gana casi todo de grado 1 a 2 (4.74 a 4.03), de ahí se queda en 4.0 y en grado 8 ya sube un poco.
MiBici sigue bajando hasta grado 8 (8.36), porque la forma tiene dos picos y hay 4,500 horas para sostener la curva,
pero julio queda unas 2 bicis arriba de train en todos los grados.

In [ ]:
largo = por_grado.melt(id_vars=["datos", "grado", "partición"], value_vars=["MAE", "RMSE", "R²"], var_name="vara")
g = sns.relplot(data=largo, x="grado", y="value", hue="partición", row="datos", col="vara", kind="line",
                marker="o", height=2.8, aspect=1.4, facet_kws={"sharey": False},
                palette={"train": COLORES["w"], "validación": COLORES["error"]})
g.set_titles("{row_name}: {col_name}")
g.set_axis_labels("grado", "")
plt.show()

**¿Qué esperamos ver?** Las líneas azules (train) siempre bajan en RMSE y suben en R²; el MAE casi siempre, porque la regresión minimiza
el error cuadrático, no el absoluto (en MiBici sube un pelo de grado 2 a 3). Las rojas (validación) en autos se aplanan desde grado 2:
R² pasa de 0.62 a 0.73 y ahí se queda. Train siempre mejora, validación no: subir el grado solo vale la pena mientras la roja baje.

## 6. Anscombe: mismos números, distintos datos

Cuatro datos de 11 puntos, armados en 1973 por Frank Anscombe. Les ajustamos una recta a cada uno y medimos su R².

**Si el R² es igual, ¿los datos se parecen?**

In [ ]:
anscombe = externo("anscombe")

r2 = {}
for nombre, grupo in anscombe.groupby("dataset"):
    ajuste = LinearRegression().fit(grupo[["x"]], grupo.y)
    r2[nombre] = r2_score(grupo.y, ajuste.predict(grupo[["x"]]))

g = sns.lmplot(data=anscombe, x="x", y="y", col="dataset", col_wrap=2, ci=None, height=3.2,
               scatter_kws={"color": COLORES["w"], "s": 50}, line_kws={"color": COLORES["error"]})
for ax, nombre in zip(g.axes.flat, g.col_names):
    ax.set_title(f"Anscombe {nombre}: R² {r2[nombre]:.2f}")
plt.show()
pd.Series(r2, name="R²").round(3).to_frame().T

**¿Qué esperamos ver?** R² de 0.67 en los cuatro. Las gráficas no se parecen: I es una nube que sube, II es una curva, III es una línea
perfecta con un punto fuera que jala la recta y IV es una columna con un solo punto lejos que decide toda la pendiente.
Un número no basta: siempre hay que mirar la gráfica.

## 7. Para pensarlo como analistas

1. **Operaciones de MiBici pide un solo número para saber qué tan confiable es el pronóstico. ¿Cuál de las tres varas
   le reportas y en qué partición?** Piensa en quién lo va a leer: un "nos equivocamos por unas X bicis por hora" se entiende
   mejor que un R².
2. **¿Qué grado usarías para cada dato y por qué?** Mira la validación, no train, y prefiere el grado más chico
   que ya no mejora al subirlo.
3. **Si los residuos de tu modelo tienen forma, ¿qué haces?** La forma dice qué le falta al modelo: curva, otra variable
   (¿el día de la semana?, ¿la lluvia?) o separar los datos en grupos.